# Data Cleaning Pipeline
Takes in a single netcdf file and a parameter list/value array, cleans off the spinup time and removes values that we do not want the neural network to see.

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import re

In [2]:
## Config block, change things here
SCRATCH_DIR = Path("/scratch/alpine/dagr5998/ksno2/")
forcing_loc_list = ["centralarctic"]
saved_vars = ['evap','fsnow','frazil','fswabs','flw','flwout','fsens','fsurf','flat','frain','Tair','Qa','fsw','fcondtop','meltt','meltb','meltl','snoice','dsnow','congel','sst','sss','Tf','fhocn','melts', 'aicen','vicen', 'vsnon']
saved_tracers_dict = {"Tsfcn":[1],"qice":[2,3,4,5,6,7,8,9],"qsno": [10, 11, 12], "sice":[13,14,15,16,17,18,19,20]} #using fortran/.sel indexing, check log file for tracer defs
start_year = 2000
spinup_years=1
OUT_DIR = Path("/projects/dagr5998/icepack_array_output/test2_output")
agg_hours = 24

In [3]:
#modfile = SCRATCH_DIR/modfile_name / f"{modfile_name}.mods"
#ENSEMBLE_DIR = SCRATCH_DIR / modfile_name

In [4]:
def read_mod_file(modfile):
    def parse(val):
        val = val.strip().rstrip(",")
        low = val.lower()
        if low in (".true.", ".false."):
            return low == ".true."
        try:
            return float(low.replace("d", "e"))
        except ValueError:
            return val.strip("'\"")          # strings stay strings

    param_dict = {}
    for line in modfile.read_text().splitlines():
        line = line.split("!")[0].strip()
        if line == '' or "=" not in line:
            continue
        k, v = line.split("=", 1)
        param_dict[k.strip()] = parse(v)
    return param_dict

In [5]:
def clean_run(path, saved_vars, saved_tracers, spinup_years=2):
    # Takes the path to the dataset and basic inputs and retearns cleaned dataset
    with xr.open_dataset(path, decode_times=False) as ds:
        ds = ds.sel(ni=3)

        out = {v: ds[v] for v in saved_vars}
        for name, idxs in saved_tracers.items():
            for i in idxs:
                out[f"{name}{i}"] = ds["trcrn"].sel(ntrcr=i).drop_vars("ntrcr")

        out = xr.Dataset(out).drop_vars("ni").load()

    return out.isel(time=slice(spinup_years * 365 * 24, None))

In [6]:
def aggregate(ds, hours=24):
    #cuts down on memory, keeps trend
    if hours == 1:
        return ds
    return ds.coarsen(time=hours, boundary="trim").mean()

In [7]:
def combine_ensemble(ENSEMBLE_DIR, saved_vars, saved_tracers_dict , forcing_loc, modfile):
    #takes in the directory of the ensemble and modfile path, returns cleaned xarray and mod parameters/vals dict
    pat = re.compile(rf"{forcing_loc}_forced_(\d+)_(\d+)-(\d+)$") #uses regex to determine location, ensemble members and date
    runs, members, windows = [], [], []
    for RUN_DIR in sorted(ENSEMBLE_DIR.glob(f"{forcing_loc}_forced_*")):
        m = pat.match(RUN_DIR.name)
        mem, y0 = m.group(1), m.group(2)
        
        nc = RUN_DIR / "history" / f"icepack.h.{y0}0101.nc"
        ds_i = aggregate(clean_run(nc, saved_vars, saved_tracers_dict, spinup_years), agg_hours)
        runs.append(ds_i.drop_vars("time"))
        members.append(mem)
        windows.append(y0)

    clean_ds = xr.concat(runs, dim="run")
    clean_ds = clean_ds.assign_coords(run=np.arange(len(runs)))

    return (clean_ds, read_mod_file(modfile), np.array(members, dtype=np.int32), np.array(windows, dtype=np.int32))

In [8]:
def check_nans(ds, default=-9999):
    # check for any nans, infs, or unset values
    print(f"{'variable':12s} {'nan':>10s} {'default':>10s} {'inf':>10s} {'total':>12s}")
    for name, da in ds.data_vars.items():
        a = da.values
        n_nan  = int(np.isnan(a).sum())
        n_sent = int(np.isclose(a, default).sum())
        n_inf  = int(np.isinf(a).sum())
        flag = "  <--" if (n_nan or n_sent or n_inf) else ""
        print(f"{name:12s} {n_nan:>10d} {n_sent:>10d} {n_inf:>10d} {a.size:>12d}{flag}")

In [9]:
def to_array(ds, saved_vars, saved_tracers_dict):
    #Takes in a cleaned xarray netcdf dataset and the pulled variables and puts it into (N, T, C): (Number of runs, Timesteps, Channel (variables)) 
    names = list(saved_vars) + [f"{n}{i}" for n, index in saved_tracers_dict.items() for i in index]
    #print(names)
    ncat = ds.sizes["ncat"]
    cols, channels = [], []
    for i in names:
        ds_i = ds[i]
        if "ncat" in ds_i.dims:
            cols.append(ds_i.transpose("run", "time", "ncat").values)
            channels += [f"{i}_ncat{c}" for c in range(1, ncat + 1)]
        else:
            cols.append(ds_i.transpose("run", "time").values[:, :, None])
            channels += [i]
    return np.concatenate(cols, axis=2).astype(np.float32), channels

In [10]:
Xarray_list , Yarray_list , group_list , member_list, window_list = [],[],[],[],[]
params_names = None
for g, ENSEMBLE_DIR in enumerate(sorted(SCRATCH_DIR.glob("*"))):
    modfile_name = ENSEMBLE_DIR.name
    modfile = ENSEMBLE_DIR/ f"{modfile_name}.mods"
    
    ds, params, member_i, window_i = combine_ensemble(ENSEMBLE_DIR, saved_vars, saved_tracers_dict, forcing_loc_list[0], modfile)
    X_i, channels = to_array(ds, saved_vars, saved_tracers_dict)
    
    names_i = [k for k, v in params.items() if isinstance(v, float)]
    if params_names is None:
        params_names = names_i
    assert names_i == params_names, (
        f"{modfile_name}: numeric params {names_i} != {params_names}")
    
    Xarray_list.append(X_i) 
    Yarray_list.append(np.tile([params[k] for k in params_names], (X_i.shape[0], 1))) #keeping like this so when the number of parameters increases it should still work
    group_list.append(np.full(X_i.shape[0], g+1)) # same group for any one parameter value set (30 ensemble members in 1 group)
    member_list.append(member_i) # ensemble member
    window_list.append(window_i) # window is whatever forcing years it was run on
    
X = np.concatenate(Xarray_list) # np array of the form (N, T, C), has the values of the times series for all ensemble members and "channels"
Y = np.concatenate(Yarray_list) # np array of the form (N, P) P is the parameters changed, has the values of the parameters for each ensemble member
group  = np.concatenate(group_list) #np array of the form (N,), keeps track of the parameter value set everything was run on
member = np.concatenate(member_list) #np array of the form (N,), keeps track of which forcing it was run on (the model shouldn't see this!)
window = np.concatenate(window_list) #np array of the form (N,), keeps track of first year run
print(X.shape, Y.shape, group.shape, member.shape, window.shape)

(140, 730, 140) (140, 1) (140,) (140,) (140,)


In [11]:
window

array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 2012, 2013, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007,
       2008, 2009, 2010, 2011, 2012, 2013, 2000, 2001, 2002, 2003, 2004,
       2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2000, 2001,
       2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
       2013, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009,
       2010, 2011, 2012, 2013, 2000, 2001, 2002, 2003, 2004, 2005, 2006,
       2007, 2008, 2009, 2010, 2011, 2012, 2013, 2000, 2001, 2002, 2003,
       2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2000,
       2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011,
       2012, 2013, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 2013, 2000, 2001, 2002, 2003, 2004, 2005,
       2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013], dtype=int32)

In [12]:
np.save(OUT_DIR / "X.npy", X)
np.save(OUT_DIR/ "Y.npy", Y)
np.save(OUT_DIR / "group.npy", group)
np.save(OUT_DIR/ "member.npy", member)
np.save(OUT_DIR/ "window.npy", window)
config = {
    "channels":     channels,
    "param_names":  params_names,
    "shape":        list(X.shape),
    "spinup_years": spinup_years,
    "source_dir":   str(SCRATCH_DIR),
    "saved_vars":    saved_vars,
    "saved_tracers": saved_tracers_dict,
    "aggregated hours": agg_hours,
    "created":       datetime.now().isoformat(timespec="seconds"),
    "notes": {
        "tracer_naming": "Tracer channel numbers are Fortran ntrcr indices from the run log, not layer numbers. qice2-qice9 are ice layers 1-8 are so forth ",
    },
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2)) #creates a record to go along with the output


3641